# UTS Graf Sains Data

| Nama | NIM | Kelas |
|---|---|---|
| Muh. Hilmy Fahrizal | 23031554125 | 2023D |

### 1.3 Impor Data RetailRocket ke Neo4j

Sumber data: Roman Zykov (Kaggle, `retailrocket/ecommerce-dataset`). File yang digunakan adalah `events.csv`, diunduh dengan perintah `curl` pada soal dan diletakkan di folder `data/`.

#### Isi data

| Kolom | Arti |
|---|---|
| `timestamp` | waktu kejadian, unix epoch dalam milidetik |
| `visitorid` | ID unik pengunjung |
| `event` | jenis interaksi: `view`, `addtocart`, atau `transaction` |
| `itemid` | ID unik produk |
| `transactionid` | ID transaksi, hanya terisi saat `event` = `transaction` |

#### Model graf

Setiap baris data menjadi **satu relasi berarah** dari pengunjung ke produk. Jenis relasinya ditentukan oleh kolom `event`:

| Nilai `event` | Relasi di Neo4j |
|---|---|
| `view` | `(:Visitor)-[:VIEWED]->(:Item)` |
| `addtocart` | `(:Visitor)-[:ADDED_TO_CART]->(:Item)` |
| `transaction` | `(:Visitor)-[:BOUGHT]->(:Item)` |

Setiap relasi menyimpan `timestamp`, dan relasi `BOUGHT` menyimpan juga `transactionId`.

#### Alur impor

1. Membaca argumen koneksi (`argparse`).
2. Cek file `events.csv`.
3. Koneksi ke Neo4j.
4. Membuat *constraint* unik.
5. Mengimpor data per batch.

#### Langkah 1: Parameter koneksi (`argparse`)

Parameter koneksi dikumpulkan dengan `argparse` agar nilainya mudah diganti tanpa mengubah kode:

| Argumen | Nilai default | Fungsi |
|---|---|---|
| `-u`, `--user` | `neo4j` | username Neo4j | 
| `-p`, `--password` | variabel env `NEO4J_PASSWORD`| password Neo4j |
| `-b`, `--bolt` | `neo4j://127.0.0.1:7687` | alamat server (URI Bolt) |
| `-s`, `--source` | `data` | folder `events.csv` berada |

Dua penyesuaian terhadap kode pada soal:
- `parse_args([])` memakai list kosong agar `argparse` tidak membaca argumen internal Jupyter, sehingga yang digunakan adalah nilai *default*.
- Jika `NEO4J_PASSWORD` tidak diset, password diminta lewat `getpass`.

In [ ]:
import os
import argparse
import getpass

def parse_arguments():
    parser = argparse.ArgumentParser(
        description="Impor dataset RetailRocket (events.csv) ke database Neo4J."
    )
    parser.add_argument(
        "-u", "--user", default="neo4j", help="Username Neo4j (default: neo4j)"
    )
    parser.add_argument(
        "-p",
        "--password",
        default=os.getenv("NEO4J_PASSWORD"),
        help="Password Neo4J (default: variabel env NEO4J_PASSWORD)",
    )
    parser.add_argument(
        "-b",
        "--bolt",
        default="neo4j://127.0.0.1:7687",
        help="Neo4J URI / Bolt (default: neo4j://127.0.0.1:7687)",
    )
    parser.add_argument(
        "-s",
        "--source",
        default="data",
        help="Direktori/folder sumber data (default: data)",
    )
    return parser.parse_args([])

args = parse_arguments()
if args.password is None:
    args.password = getpass.getpass("Password Neo4j: ")

print({k: v for k, v in vars(args).items() if k != "password"})


{'user': 'neo4j', 'bolt': 'neo4j://127.0.0.1:7687', 'source': 'data'}


#### Langkah 2: Memeriksa berkas `events.csv`

Lokasi file dibentuk dari argumen `--source`. Jika berkas tidak ada maka akan berhenti (`sys.exit(1)`) agar tidak gagal di tengah proses impor. Jika berkas ada, sel ini tidak mengeluarkan output.

In [4]:
import sys

csv_file_path = os.path.join(args.source, "events.csv")

if not os.path.exists(csv_file_path):
    print(f"Error: File '{csv_file_path}' tidak ditemukan!")
    sys.exit(1)


#### Langkah 3: Koneksi ke Neo4j

`GraphDatabase.driver(...)` membuat objek koneksi, lalu `verify_connectivity()` memastikan server dapat disambungkan dan kredensialnya benar. Jika gagal, error muncul dan program berhenti. Jika tidak ada output apa pun berarti koneksi berhasil.

In [5]:
from neo4j import GraphDatabase

try:
    driver = GraphDatabase.driver(
        args.bolt, auth=(args.user, args.password)
    )
    driver.verify_connectivity()
except Exception as e:
    print(f"Gagal terhubung ke database: {e}")
    sys.exit(1)


#### Langkah 4: *Constraint* unik

Sebelum mengimpor, dibuat *constraint* keunikan pada `Visitor.visitorId` dan `Item.itemId`. Constraint ini berfungsi:
- **Integritas:** tidak mungkin ada dua simpul `Visitor` (atau `Item`) dengan ID yang sama.
- **Kecepatan:** `MERGE` mencari simpul lewat *index* yang otomatis dibuat bersama constraint.

In [6]:
def create_constraints(session):
    session.run(
        "CREATE CONSTRAINT visitor_id IF NOT EXISTS "
        "FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE"
    )
    session.run(
        "CREATE CONSTRAINT item_id IF NOT EXISTS "
        "FOR (i:Item) REQUIRE i.itemId IS UNIQUE"
    )


#### Langkah 5: Query impor (`import_batch`)

Fungsi ini menerima satu batch baris data dan memprosesnya dalam satu transaksi:

| Bagian query | Fungsi |
|---|---|
| `UNWIND $batch AS row` | memecah list batch menjadi satu iterasi per baris data |
| `MERGE (v:Visitor {visitorId: row.visitorid})` | membuat simpul visitor jika belum ada |
| `MERGE (i:Item {itemId: row.itemid})` | membuat simpul item jika belum ada |
| `CALL { ... UNION ALL ... }` | membuat relasi sesuai `row.event`, dengan cabang `WHERE row.event = ...` |

Untuk tiap baris data hanya satu cabang yang cocok, sehingga setiap baris menghasilkan tepat satu relasi (`VIEWED`, `ADDED_TO_CART`, atau `BOUGHT`).

Simpul dibuat dengan `MERGE` agar tidak ganda, sedangkan relasi dibuat dengan `CREATE` karena satu pengunjung dapat berinteraksi berkali-kali dengan item yang sama, dan setiap kejadian perlu tersimpan dengan `timestamp`-nya sendiri.

In [7]:
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})

    WITH v, i, row
    CALL {
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)


#### Langkah 6: Impor per batch

Berkas dibaca per batch dengan `pd.read_csv(..., chunksize=1000)`. Setiap batch diproses dengan urutan berikut:

1. Nilai kosong (`NaN`) pada `transactionid` diganti string kosong agar tidak ada `NaN` yang dikirim ke Neo4j.
2. Batch diubah menjadi list *dictionary* (`to_dict(orient="records")`).
3. `session.execute_write(import_batch, records)` menjalankan satu transaksi untuk batch tersebut.

`tqdm` menampilkan kemajuan; jumlah batch (`total_chunks`) dihitung dari jumlah baris berkas (tanpa header) dibagi `batch_size`.

**Pengaman impor ulang.** Selama `IMPOR_ULANG = False`, impor dilewati jika graf sudah berisi interaksi, sehingga relasi tidak menjadi ganda ketika notebook dijalankan ulang. Ubah menjadi `True` hanya setelah graf dikosongkan (`MATCH (n) DETACH DELETE n`).

In [ ]:
from tqdm import tqdm
import pandas as pd

batch_size = 1000
IMPOR_ULANG = False

with driver.session() as session:
    create_constraints(session)

    sudah_ada = session.run(
        "MATCH ()-[r:VIEWED|ADDED_TO_CART|BOUGHT]->() WITH r LIMIT 1 RETURN count(r) AS n"
    ).single()["n"]

    if sudah_ada and not IMPOR_ULANG:
        print("Graf sudah berisi data interaksi, impor dilewati.")
    else:
        print(f"Membaca file data dari: {csv_file_path}")
        chunks = pd.read_csv(csv_file_path, chunksize=batch_size)

        with open(csv_file_path) as f:
            total_rows = sum(1 for _ in f) - 1          # tanpa baris header
        total_chunks = (total_rows // batch_size) + 1

        print("Memulai proses impor data ke Neo4j...")
        for chunk in tqdm(
            chunks, total=total_chunks, desc="Mengimpor Batch Data"
        ):
            # Membersihkan data null/NaN pada transactionid
            chunk["transactionid"] = (
                chunk["transactionid"].fillna("").astype(str)
            )
            records = chunk.to_dict(orient="records")
            session.execute_write(import_batch, records)

        print("Proses impor berhasil.")


Membaca file data dari: data\events.csv
Memulai proses impor data ke Neo4j...


Mengimpor Batch Data:   1%|          | 15/2757 [00:03<05:17,  8.63it/s]Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (v, i, row) { ... }', position=<SummaryInputPosition line=7, column=5, offset=139>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 139, 'line': 7, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    UNWIND $batch AS row\n    MERGE (v:Visitor {visitorId: row.visitorid})\n    MERGE (i:Item {itemId: row.itemid})\n\n    WITH v, i, row\n    CALL {\n        WITH v, i, row\n        WITH v, i, row WHERE row.event = 'view'\n        CREATE (v)-[:VIEWED {timestam

Proses impor berhasil.


**Hasil.** Impor memproses 2.757 batch hingga muncul *Proses impor berhasil*.

Peringatan *deprecation* pada output berasal dari sintaks `CALL { }` pada kode soal. Peringatan ini tidak memengaruhi hasil impor.

Jika sel ini dijalankan lagi, keluarannya adalah *"Graf sudah berisi data interaksi, impor dilewati."*

Hasil impor diverifikasi pada notebook jawaban (bagian 2.1 no. 3): graf berisi 1.407.580 `Visitor`, 235.061 `Item`, dan 2.756.101 relasi, sesuai jumlah baris `events.csv`.